Breast Cancer Classification using ANN [Optuna]


In [3]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [5]:
df = pd.read_csv('data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [6]:
df.shape

(569, 33)

In [7]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
df.shape

(569, 31)

In [9]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0] , test_size=0.2)


In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
X_train

array([[-1.16934716, -0.29710868, -1.10025629, ..., -0.2210672 ,
        -0.47169209,  1.76099193],
       [-0.05705346, -0.83623148, -0.02965844, ...,  0.38116998,
        -0.51280513,  1.03334185],
       [ 0.27697724, -0.5607326 ,  0.21032459, ..., -0.43999271,
        -0.81704157, -1.10109836],
       ...,
       [ 0.34835133,  2.68587915,  0.50792012, ...,  1.87600868,
         1.95397688,  3.12257957],
       [ 1.09920668, -1.44422899,  0.97213599, ..., -0.4621157 ,
        -1.83828932, -1.40814464],
       [-0.13413747, -1.98810177, -0.13410701, ..., -0.60560793,
        -0.31381804, -0.68049457]])

In [12]:
y_train

176    B
73     M
169    B
377    B
55     B
      ..
355    B
41     M
562    M
491    B
76     B
Name: diagnosis, Length: 455, dtype: object

In [13]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [14]:
from torch.utils.data import Dataset , DataLoader
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [15]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [16]:
class MyNN(nn.Module):
    def __init__(self, input_data, output_data, num_hidden_layers, neuron_per_layer, dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_data, neuron_per_layer))
            layers.append(nn.BatchNorm1d(neuron_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_data = neuron_per_layer  # Corrected variable

        layers.append(nn.Linear(neuron_per_layer, output_data))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [17]:
def objective(trial):
    # Suggest hyperparameters
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 10)
    neuron_per_layer = trial.suggest_int('neuron_per_layer', 8, 256, step=8)
    epochs = trial.suggest_int('epochs', 10, 100, step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # Loaders (make sure train_dataset and test_dataset are defined)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Define input/output dimensions
    # Correct the input_dim to match the actual number of features (30)
    input_dim = X_train.shape[1] # Get the number of features directly from the processed data
    output_dim = 2

    # Define model and loss
    model = MyNN(input_data=input_dim, output_data=output_dim,
                 num_hidden_layers=num_hidden_layers,
                 neuron_per_layer=neuron_per_layer,
                 dropout_rate=dropout_rate)

    criterion = nn.CrossEntropyLoss()

    # Optimizer
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Training loop
    model.train()
    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [18]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [19]:
import optuna
study = optuna.create_study(direction='maximize')


[I 2025-06-11 10:05:06,695] A new study created in memory with name: no-name-990393f8-49d3-488f-9151-5316e8a7cdad


In [20]:
study.optimize(objective, n_trials=10)

[I 2025-06-11 10:05:07,925] Trial 0 finished with value: 98.24561403508773 and parameters: {'num_hidden_layers': 1, 'neuron_per_layer': 192, 'epochs': 30, 'learning_rate': 0.0035346704802915753, 'dropout_rate': 0.5, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 4.299059728047115e-05}. Best is trial 0 with value: 98.24561403508773.
[I 2025-06-11 10:05:09,700] Trial 1 finished with value: 99.12280701754386 and parameters: {'num_hidden_layers': 9, 'neuron_per_layer': 128, 'epochs': 20, 'learning_rate': 0.01730899519894454, 'dropout_rate': 0.5, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.00015304088208365743}. Best is trial 1 with value: 99.12280701754386.
[I 2025-06-11 10:05:12,253] Trial 2 finished with value: 58.771929824561404 and parameters: {'num_hidden_layers': 10, 'neuron_per_layer': 24, 'epochs': 90, 'learning_rate': 0.0003127993504371943, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 0.00019096327209559057}. Best is trial 1 w

In [21]:
# ...existing code...

# 1. Get the best trial's parameters
best_params = study.best_trial.params

# 2. Rebuild the best model
input_dim = X_train.shape[1]
output_dim = 2
best_model = MyNN(
    input_data=input_dim,
    output_data=output_dim,
    num_hidden_layers=best_params['num_hidden_layers'],
    neuron_per_layer=best_params['neuron_per_layer'],
    dropout_rate=best_params['dropout_rate']
)

# 3. Train the best model on the full training data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)
criterion = nn.CrossEntropyLoss()
if best_params['optimizer'] == 'Adam':
    optimizer = optim.Adam(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
elif best_params['optimizer'] == 'SGD':
    optimizer = optim.SGD(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
elif best_params['optimizer'] == 'RMSprop':
    optimizer = optim.RMSprop(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)
epochs = best_params['epochs']

best_model.train()
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = best_model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# 4. Prediction function
def predict_cancer(input_features):
    """
    input_features: list or numpy array of shape (30,)
    Returns: 'Malignant' or 'Benign'
    """
    # Scale input
    input_scaled = scaler.transform([input_features])
    input_tensor = torch.tensor(input_scaled, dtype=torch.float32).to(device)
    best_model.eval()
    with torch.no_grad():
        output = best_model(input_tensor)
        _, predicted = torch.max(output.data, 1)
        label = encoder.inverse_transform(predicted.cpu().numpy())[0]
    return f"Cancer Detected: {label}"

# Example usage:
# Replace the below with your own 30-feature input
sample_input = X_test[0]  # or your own data
print(predict_cancer(sample_input))

Cancer Detected: M


/Users/tanmayjaipuriar/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
